## RAG pipeline - Data Ingestion to Vector DB

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/kv/f0xmp3wd0n51ydqcckq6wpyh0000gn/T/ipykernel_5849/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


## Data Ingestion

In [16]:
## Read all pdfs inside the directory
def process_all_pdfs(dir:str):
    """"Process all the pdfs inside the directory and return the list of documents"""
    all_documents=[]
    pdf_dir = Path(dir)

    #Find all pdfs recursively inside the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files in the directory {dir}")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        loader = PyPDFLoader(pdf_file)
        documents = loader.load() # N documents for N pages in the pdf
        print(f"+ Loaded: {len(documents)}")
        
        # Add source metadata to each document
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"]   = "pdf"

        all_documents.extend(documents)
    print(f"Processed {len(all_documents)} documents from {len(pdf_files)} pdf files")
    return all_documents

In [17]:
all_pdf_documents = process_all_pdfs("../data")

Found 6 pdf files in the directory ../data
Processing: python_basics.pdf
+ Loaded: 5
Processing: data_engineering.pdf
+ Loaded: 5
Processing: software_design.pdf
+ Loaded: 5
Processing: machine_learning_intro.pdf
+ Loaded: 5
Processing: rag_systems.pdf
+ Loaded: 6
Processing: aws_cloud.pdf
+ Loaded: 5
Processed 31 documents from 6 pdf files


## Data Chunking

In [18]:
def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks of specified size with overlap"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    chunks = text_splitter.split_documents(all_pdf_documents)
    print(f"Splited {len(documents)} documents into {len(chunks)} chunks")

    # Show example of a chunk
    if chunks:
        print("\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:200]}...")
        print(f"Metadata: {chunks[0].metadata}")
    return chunks


In [19]:
chunks = chunk_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Splited 31 documents into 197 chunks

Example chunk:
Content: Python Basics
Section 1
Python Basics discusses concepts, examples, best practices, and practical applications. Lorem
ipsum dolor sit amet, consectetur adipiscing elit. Praesent efficitur, sapien at v...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-04T11:05:21+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-04T11:05:21+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/python_basics.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'python_basics.pdf', 'file_type': 'pdf'}


## Embedding and storing into vector store database

In [20]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingGenerator:
    "Handle document embedding generation using Sentence Transformers"
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding generator
        Args:
            model_name (str): Hugging Face model name for sentence transformers. Default is all-MiniLM-L6-v2
            """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension is: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts (List[str]): List of text strings to generate embeddings for"""
        if not self.model:
            raise ValueError("Model is not loaded.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [22]:
embedding_manager = EmbeddingGenerator()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6243.51it/s]


Model loaded successfully. Embedding dimension is: 384


/var/folders/kv/f0xmp3wd0n51ydqcckq6wpyh0000gn/T/ipykernel_5849/2235720056.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension is: {self.model.get_sentence_embedding_dimension()}")


#### Vector Store using ChromaDB

In [23]:
class VectorStore:
    """Handle vector store operations using ChromaDB"""
    def __init__(self, collection_name: str= "pdf_documments", persist_directory: str = "../data/chroma_db"):
        """Initilize the vector store
        Args:
            collection_name (str): Name of the collection in ChromaDB. Default is "pdf_documments"
            persist_directory (str): Directory to persist the ChromaDB database. Default is "../data/chroma_db"
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # create perisist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # create collection if not exists
            self.collection = self.client.get_or_create_collection(name=self.collection_name, 
                                                                   metadata={"description": "PDF documents embedding for RAG"})
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing collection size: {self.collection.count()}")
        except Exception as e:
            ValueError(f"Error occurred while initializing the vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
            documents (List[Any]): List of langchain documents
            embeddings (np.ndarray): Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # document text
            documents_texts.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        try:
            # add to collection
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total collection size after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error occurred while adding documents to the vector store: {e}")
            raise

In [24]:
vector_store = VectorStore()

Vector store initialized with collection: pdf_documments
Existing collection size: 0


In [25]:
# convert text to embedding
# First we need to extract the text from the documents
texts = [doc.page_content for doc in chunks]

# Generate embeddings for the extracted texts
embeddings = embedding_manager.generate_embeddings(texts)

# store the documents and embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Batches: 100%|██████████| 7/7 [00:03<00:00,  2.17it/s]

Generated embeddings with shape: (197, 384)
Adding 197 documents to the vector store...
Successfully added 197 documents to the vector store.
Total collection size after addition: 197
